# Intro

**Seminar by @Elad_Benjo**

In this notebook I will attempt to develop a new model for nowcsating of the Israeli GDP.

Our efforts would be towards a DFM, since we have various frequencies and high demensional data.

In [1]:
from google.colab import drive
import sys

drive.mount('/content/drive')
sys.path.append('/content/drive/MyDrive/gdp_nowcasting_seminar/src')

Mounted at /content/drive


## Papers

https://www.federalreserve.gov/econres/ifdp/files/ifdp1385.pdf

# Pre-model Prep

In [ ]:
import pandas as pd

In [ ]:
daily_df = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/feature_selected/daily_data.pkl')

In [ ]:
monthly_df = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/feature_selected/monthly_data.pkl')

In [ ]:
monthly_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 364 entries, 1995-01-01 to 2025-04-01
Freq: MS
Data columns (total 43 columns):
 #   Column                                                                          Non-Null Count  Dtype  
---  ------                                                                          --------------  -----  
 0   Total refunds from the Income Tax Department                                    339 non-null    float64
 1   self employed returns                                                           339 non-null    float64
 2   Cancellations Deductions                                                        352 non-null    float64
 3   Capital Gas Tax Refunds                                                         351 non-null    float64
 4   Cancellation companies                                                          352 non-null    float64
 5   Purchase returns                                                                351 non-null    flo

0, 1, 2, 3, 4, 5, 6, 7, 10, 16, 19, 20, 21, 23, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42

In [ ]:
monthly_df.iloc[:, [0, 1, 2, 3, 4, 5, 6, 7, 10, 16, 19, 20, 21, 23, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42]] = monthly_df.iloc[:, [0, 1, 2, 3, 4, 5, 6, 7, 10, 16, 19, 20, 21, 23, 24, 25, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42]].shift(1)

In [ ]:
import preprocessing

In [ ]:
df_X = preprocessing.merge_series_freq({'d': daily_df, 'm': monthly_df})

In [ ]:
df_X.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9482 entries, 1995-01-01 to 2025-08-01
Data columns (total 47 columns):
 #   Column                                                                            Non-Null Count  Dtype  
---  ------                                                                            --------------  -----  
 0   d_TA125                                                                           4879 non-null   float64
 1   d_SP500                                                                           6135 non-null   float64
 2   d_USD_ILS                                                                         5969 non-null   float64
 3   d_SPGSCI                                                                          6135 non-null   float64
 4   m_Total refunds from the Income Tax Department                                    339 non-null    float64
 5   m_self employed returns                                                           339 non-nul

In [ ]:
df_X = df_X.asfreq("D")

In [ ]:
print(df_X.index.freq)

<Day>


In [ ]:
df_X.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 11171 entries, 1995-01-01 to 2025-08-01
Freq: D
Data columns (total 47 columns):
 #   Column                                                                            Non-Null Count  Dtype  
---  ------                                                                            --------------  -----  
 0   d_TA125                                                                           4879 non-null   float64
 1   d_SP500                                                                           6135 non-null   float64
 2   d_USD_ILS                                                                         5969 non-null   float64
 3   d_SPGSCI                                                                          6135 non-null   float64
 4   m_Total refunds from the Income Tax Department                                    339 non-null    float64
 5   m_self employed returns                                                           33

## Publication Shifts

Tax information is relead in a 10 days delay

In [ ]:
df_X.iloc[:, [4, 5, 6, 7, 8, 9, 10, 11, 14, 20, 23, 24, 25, 27, 28, 29, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46]] = df_X.iloc[:, [4, 5, 6, 7, 8, 9, 10, 11, 14, 20, 23, 24, 25, 27, 28, 29, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46]].shift(10)

CPI is 15 days later  
Consumer Trust Index is 12 days later  
Unemployment rate is 16 days later  
Real Credit Card Usage is 3 days later


In [ ]:
df_X.iloc[:, [17]] = df_X.iloc[:, [17]].shift(15)
df_X.iloc[:, [19]] = df_X.iloc[:, [19]].shift(12)
df_X.iloc[:, [12]] = df_X.iloc[:, [12]].shift(16)
df_X.iloc[:, [21]] = df_X.iloc[:, [21]].shift(3)

real salary delay is 50 days then drop it

In [ ]:
df_X = df_X.drop(df_X.columns[26], axis=1)

drop the first 2 rows, and anything beyond 16.1.25

In [ ]:
df_X = df_X.iloc[2:]
df_X = df_X[:'2025-01-16']

## Train-Test Split

I split the data according to the model currently used by the Ministry of Finance, in order to maintain a relevant basis for performance comparison.

In [ ]:
df_X.to_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/df_X.pkl')

In [ ]:
x_train = df_X[:'2022-01-16']
x_test = df_X['2022-01-17':]

In [ ]:
x_train.to_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/x_train.pkl')

In [ ]:
x_test.to_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/x_test.pkl')

In [ ]:
df_Y = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/stationary/quarterly_data.pkl')

In [ ]:
df_Y.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 119 entries, 1995-06-30 to 2024-12-31
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   GDP     119 non-null    float64
 1   C       119 non-null    float64
 2   G       115 non-null    float64
 3   I       119 non-null    float64
 4   EX      119 non-null    float64
 5   IM      119 non-null    float64
dtypes: float64(6)
memory usage: 6.5 KB


In [ ]:
df_Y.tail(20)

,GDP,C,G,I,EX,IM
Date,,,,,,
2020-03-31,-0.020815,-0.059381,-0.032948,0.005003,-0.030563,-0.066587
2020-06-30,-0.086301,-0.122846,0.013079,-0.140037,-0.085846,-0.114879
2020-09-30,0.085868,0.080171,0.025859,0.059847,0.102935,-0.008056
2020-12-31,0.025544,0.044362,0.024968,0.132744,0.026366,0.169621
2021-03-31,0.004977,0.004431,0.007891,-0.018693,0.040784,0.029436
2021-06-30,0.034456,0.063183,-0.001098,0.033092,0.024298,0.044995
2021-09-30,0.019257,0.013908,-0.026294,0.034772,0.038396,0.033290
2021-12-31,0.043839,0.034346,0.010034,0.039539,0.064620,0.065860
2022-03-31,-0.006193,0.010369,-0.014388,0.031372,-0.001565,0.033355


In [ ]:
y_train = df_Y[:'2021-12-31']
y_test = df_Y['2022-01-01':]

In [ ]:
y_train.to_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/y_train.pkl')

In [ ]:
y_test.to_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/y_test.pkl')

In [ ]:
x_train.tail()

,d_TA125,d_SP500,d_USD_ILS,d_SPGSCI,m_Total refunds from the Income Tax Department,m_self employed returns,m_Cancellations Deductions,m_Capital Gas Tax Refunds,m_Cancellation companies,m_Purchase returns,...,m_Goods and services,m_Independents advances,m_Self-employed tax differences,m_Non-profit institution tax,m_tax differential companies,m_Companies advances,m_Income tax for self-employed dividuals and companies (advances and deductions),m_Deduction from salary,m_Deductions and the capital market,m_Total Income Tax Division Net
Date,,,,,,,,,,,,,,,,,,,,,
2022-01-12,0.015241,0.002814,-0.004608,0.012775,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-13,0.006246,-0.014346,0.001806,-0.009244,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-14,NaN,0.000820,-0.001726,0.011274,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
y_train.tail()

,GDP,C,G,I,EX,IM
Date,,,,,,
2020-12-31,0.025544,0.044362,0.024968,0.132744,0.026366,0.169621
2021-03-31,0.004977,0.004431,0.007891,-0.018693,0.040784,0.029436
2021-06-30,0.034456,0.063183,-0.001098,0.033092,0.024298,0.044995
2021-09-30,0.019257,0.013908,-0.026294,0.034772,0.038396,0.033290
2021-12-31,0.043839,0.034346,0.010034,0.039539,0.064620,0.065860


## Remove empty rows

In [ ]:
import pandas as pd

In [ ]:
df_X_train = pd.read_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/df_X_train.pkl')

In [ ]:
# מספר השורות הכולל
total_rows = len(x_train)

# כמה שורות הן לגמרי ריקות (כל העמודות NaN)
empty_rows = x_train.isna().all(axis=1).sum()

# אחוז שורות ריקות
empty_pct = empty_rows / total_rows * 100

print(f"Total rows: {total_rows}")
print(f"Completely empty rows: {empty_rows}")
print(f"Percentage empty: {empty_pct:.2f}%")


Total rows: 9876
Completely empty rows: 3219
Percentage empty: 32.59%


In [ ]:
# 1. remove rows that contain all NaN
df_x_train = df_X_train.dropna(how="all")

# 2. recreate continues daily index
full_index = pd.date_range(df_X_train.index.min(), df_X_train.index.max(), freq="D")
df_x_train = df_x_train.reindex(full_index)

print(df_X_train.shape, " -> ", df_x_train.shape)


(9860, 47)  ->  (9860, 47)


## Z-Score

In [ ]:
import stats_trans

In [ ]:
mu, sigma = stats_trans.fit_zscore(x_train)

In [ ]:
x_train_z_score = stats_trans.transform_zscore(x_train, mu, sigma)

In [ ]:
x_train_z_score.to_pickle('/content/drive/MyDrive/gdp_nowcasting_seminar/Data/pickles/dfm/df_x_train_z_score.pkl')

## Train DFM

In [2]:
import dfm

In [9]:
import importlib

In [10]:
importlib.reload(dfm)

<module 'dfm' from '/content/drive/MyDrive/gdp_nowcasting_seminar/src/dfm.py'>

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 9876 entries, 1995-01-03 to 2022-01-16
Freq: D
Data columns (total 46 columns):
 #   Column                                                                            Non-Null Count  Dtype  
---  ------                                                                            --------------  -----  
 0   d_TA125                                                                           4211 non-null   float64
 1   d_SP500                                                                           5418 non-null   float64
 2   d_USD_ILS                                                                         5283 non-null   float64
 3   d_SPGSCI                                                                          5418 non-null   float64
 4   m_Total refunds from the Income Tax Department                                    300 non-null    float64
 5   m_self employed returns                                                           300

,d_TA125,d_SP500,d_USD_ILS,d_SPGSCI,m_Total refunds from the Income Tax Department,m_self employed returns,m_Cancellations Deductions,m_Capital Gas Tax Refunds,m_Cancellation companies,m_Purchase returns,...,m_Goods and services,m_Independents advances,m_Self-employed tax differences,m_Non-profit institution tax,m_tax differential companies,m_Companies advances,m_Income tax for self-employed dividuals and companies (advances and deductions),m_Deduction from salary,m_Deductions and the capital market,m_Total Income Tax Division Net
Date,,,,,,,,,,,,,,,,,,,,,
1995-01-03,0.111491,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-04,-0.051108,0.269072,0.263367,-0.371298,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-05,-1.032462,-0.102563,-0.324719,0.177596,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-06,NaN,0.031231,-0.177831,-0.626882,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1995-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ===== Example usage (skeleton) =====
# X_train: daily DataFrame, stationary + Z-scored on train only

# 1) Run grid
# results_table, models = fit_dfm_grid(
#     X_train=X_train_std,
#     k_factors_grid=[1, 2, 3],
#     factor_order_grid=[1, 2],
#     error_cov_grid=["diagonal"],  # start simple
#     method="em",
#     disp=False
# )

# 2) Pick a candidate (e.g., the first row = best BIC)
# best = results_table.iloc[0]
# key = (int(best.k_factors), int(best.factor_order), best.error_cov_type)
# res_best = models[key]

# 3) Extract daily factors (use 'filtered' for real-time/nowcasting)
# F_daily = extract_factors(res_best, which="filtered")  # DataFrame with columns F1, F2, ...

# 4) (Later) Apply the trained model to new data (standardized with the same train μ,σ):
# res_on_test = res_best.apply(X_test_std)
# F_daily_test = extract_factors(res_on_test, which="filtered")

In [7]:
results_table, models = dfm.fit_dfm_grid(
    X_train=x_train_z_score,
    k_factors_grid=[1],
    factor_order_grid=[1, 2],
    error_cov_grid=["diagonal"],  # startsimple
    method="em",
    disp=False)

/usr/local/lib/python3.12/dist-packages/statsmodels/multivariate/pca.py:244: ValueWarning: The requested number of components is more than can be computed from data. The maximum number of components is the minimum of the number of observations or variables
  warnings.warn(warn, ValueWarning)
/usr/local/lib/python3.12/dist-packages/statsmodels/multivariate/pca.py:244: ValueWarning: The requested number of components is more than can be computed from data. The maximum number of components is the minimum of the number of observations or variables
  warnings.warn(warn, ValueWarning)


In [8]:
best = results_table.iloc[0]
key = (int(best.k_factors), int(best.factor_order), best.error_cov_type)
res_best = models[key]

KeyError: (1, 1, 'diagonal')

In [11]:
max_factors = dfm.max_possible_factors(x_train_z_score)
print(f"Max factors you can safely request: {max_factors}")


Max factors you can safely request: 0


In [15]:
import statsmodels.api as sm

In [17]:
mod = sm.tsa.DynamicFactor(
    x_train_z_score,
    k_factors=1,
    factor_order=1,
    error_cov_type="diagonal"
)

# Get default start_params (small positive values etc.)
start_params = mod.start_params

# Optionally: jitter them a bit, or set loadings ~ 0.1
# start_params = np.ones_like(start_params) * 0.1

res = mod.fit(start_params=start_params, method="em", disp=False)


/usr/local/lib/python3.12/dist-packages/statsmodels/multivariate/pca.py:244: ValueWarning: The requested number of components is more than can be computed from data. The maximum number of components is the minimum of the number of observations or variables
  warnings.warn(warn, ValueWarning)


ValueError: Removal of missing values has eliminated all data.